# Step 4 — ConvNeXt-Tiny with a custom segmentation head

This notebook uses synthetic coastlines and downloads no satellite data. It shows how a classification backbone becomes a pixel-wise segmentation model.

In [ ]:
%pip install -q -e ".[ml]"

## 1. Create the same input used for the UNet test
Using identical inputs makes the architectural comparison meaningful.

In [ ]:
import torch
from coastlearn.synthetic import make_synthetic_coast

examples = [make_synthetic_coast(height=128, width=128, seed=seed) for seed in (7, 11)]
images = torch.stack([torch.from_numpy(image) for image, _ in examples])
masks = torch.stack([torch.from_numpy(mask) for _, mask in examples])
print("images:", images.shape)
print("masks: ", masks.shape)

## 2. Build ConvNeXt-Tiny
`pretrained=False` avoids a weight download in this mechanics-only example. Real training will set it to `True`. The original image classifier is absent because `features_only=True` asks timm for intermediate feature maps instead.

In [ ]:
from coastlearn.models import build_convnext_tiny

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_convnext_tiny(in_channels=5, num_classes=2, pretrained=False).to(device)
images = images.to(device)
masks = masks.to(device)
print("device:", device)

## 3. Inspect the feature pyramid
Each stage uses lower spatial resolution and more channels. Resolution is traded for a larger receptive field and more semantic features. This first decoder deliberately uses only the deepest feature map.

In [ ]:
with torch.no_grad():
    feature_pyramid = model.extract_features(images)
for index, features in enumerate(feature_pyramid, start=1):
    print(f"stage {index}: {tuple(features.shape)}")

## 4. Inspect the custom head
A 3×3 convolution converts the deepest features to 256 channels. A 1×1 convolution converts those 256 values at each location into two class logits. Bilinear interpolation restores the original 128×128 resolution.

In [ ]:
with torch.no_grad():
    low_resolution_logits = model.decoder(feature_pyramid[-1])
    full_resolution_logits = model(images)
print("deepest features:     ", feature_pyramid[-1].shape)
print("before interpolation: ", low_resolution_logits.shape)
print("after interpolation:  ", full_resolution_logits.shape)

## 5. Train one batch with two learning rates
The backbone normally starts with useful pretrained features, so it receives small updates. The new decoder starts from random weights and receives larger updates.

In [ ]:
from coastlearn.training import (
    build_cross_entropy_loss,
    build_finetuning_optimizer,
    train_one_batch,
)

loss_function = build_cross_entropy_loss(ignore_index=255).to(device)
optimizer = build_finetuning_optimizer(
    model, backbone_learning_rate=1e-5, head_learning_rate=1e-3
)
print([(group["name"], group["lr"]) for group in optimizer.param_groups])
result = train_one_batch(model, images, masks, optimizer, loss_function)
print(result)

## What differs from UNet?

UNet combines several encoder resolutions through skip connections. This ConvNeXt model uses only the smallest, deepest feature map and interpolates its two-channel output. It therefore tests the backbone's semantic representation with a deliberately weak decoder. If its boundaries are coarse, the missing high-resolution skip features are a likely explanation.